# ROCLING 2026 DSA — E13：Teacher 偽標精修增強

**唯一還能推 arousal PCC 的招**（ensemble E10–E12 已證明是死路，實驗 12 官方 A_PCC 反降到 0.418）。

## 前置（已在本機完成，隨 git clone 帶入）
- `data/train_aug_pseudo.csv`：318 篇，用 3 顆 teacher（macbert_s42/s1 + roberta_s42）逐篇重標 +
  每 bin ±1.5SD 離群移除後的**校準版**合成資料。
- 對照組：實驗 4（V 0.600/0.880、A **0.882/0.426**）。目標：A_PCC 守住/提升、A_MAE 別爆。

## 已知風險（見 experiment.md 觀察 #7 + E13）
偽標 A 平均 6.42 偏高（teacher 認不出低喚醒），可能讓 A_MAE 略升；但比實驗 9 的極端 bin 標籤溫和。
**dev 對增強實驗不可信（與 dev 同風格），結論看官方提交。**

In [ ]:
# 1) 套件 + GPU 確認
!pip -q install "transformers>=4.40" jieba scikit-learn scipy
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ 無 GPU，去選 premium runtime')

In [ ]:
# 2) git clone（含 train_aug_pseudo.csv）；private repo 用 getpass 輸入 token
import os, getpass
BRANCH = 'feat/ensemble-teacher-student-experiments'
token = os.environ.get('GH_TOKEN') or getpass.getpass('GitHub token（public 直接 Enter）: ').strip()
auth = f'{token}@' if token else ''
if not os.path.exists('repo'):
    !git clone -q -b {BRANCH} https://{auth}github.com/chen0427ok/DSA-NIFT.git repo
del token, auth
os.chdir('/content/repo' if os.path.exists('/content/repo') else 'repo')
os.makedirs('outputs/preds', exist_ok=True)
assert os.path.exists('data/train_aug_pseudo.csv'), '❌ 缺 train_aug_pseudo.csv，確認已 push 到 branch'
import csv
n=len(list(csv.DictReader(open('data/train_aug_pseudo.csv'))))
print(f'✅ 就緒：偽標資料 {n} 篇')

## E13a — 主實驗：macbert + L1 + 偽標增強（單 seed，對照實驗 4）

In [ ]:
# 3) E13a：把偽標資料 append 進訓練（train.csv 本身不動，用 --extra_train）
!python train_v2.py --extra_train data/train_aug_pseudo.csv \
    --run_name macbert_pseudo_s42 --epochs 4 --batch_size 64

## E13b（可選）— 偽標增強版也跑多 seed 後平均
若 E13a 官方有進步，多 seed 平均可再穩一點（同 E10 邏輯）。額度不夠可跳過。

In [ ]:
# 4) E13b：偽標 + 3 seeds（可選）
for s in [1, 2]:
    !python train_v2.py --extra_train data/train_aug_pseudo.csv \
        --run_name macbert_pseudo_s{s} --seed {s} --epochs 4 --batch_size 64
!python ensemble.py macbert_pseudo_s42 macbert_pseudo_s1 macbert_pseudo_s2 \
    --mode mean --name e13_pseudo_ens

## 打包下載

In [ ]:
# 5) 打包 submission + 權重（走 Drive；權重太大不能推 git）
import shutil, os
zip_path = shutil.make_archive('e13_results', 'zip', 'outputs')
print('已打包 ->', zip_path, f'({os.path.getsize(zip_path)/1024**3:.2f} GB)')
# submission 小檔推回 repo（可靠）
!git config user.email "chenbrian930427@gmail.com" && git config user.name "chen0427ok"
!git add -f outputs/macbert_pseudo_*_submission.csv outputs/e13_pseudo_ens_submission.csv outputs/preds/macbert_pseudo_*.csv 2>/dev/null
!git commit -m "E13 偽標增強結果" && git push
print('submission 已推回 branch；權重在 outputs/ 需自行走 Drive 保存')

In [ ]:
# 6) 快速看預測分布（arousal 有沒有被增強資料帶偏）
import pandas as pd, glob, os
for p in sorted(glob.glob('outputs/macbert_pseudo_*_submission.csv') + glob.glob('outputs/e13_*_submission.csv')):
    df = pd.read_csv(p)
    print(f"{os.path.basename(p):34s} V {df.Valence.mean():.2f}±{df.Valence.std():.2f}  A {df.Arousal.mean():.2f}±{df.Arousal.std():.2f}")